# 3.3: A quick numerical check

The chapter's suggested code lab initializes the same deep network with small random, Xavier, and He weights and tracks activation variance by depth. A minimal version:


In [ ]:
import torch
import torch.nn as nn

def track_activation_std(init_fn, depth=30, width=256, activation=nn.ReLU):
    layers = []
    for _ in range(depth):
        lin = nn.Linear(width, width)
        init_fn(lin)
        layers += [lin, activation()]
    net = nn.Sequential(*layers)
    x = torch.randn(128, width)
    stds = []
    with torch.no_grad():
        for layer in net:
            x = layer(x)
            if isinstance(layer, activation):
                stds.append(x.std().item())
    return stds

def small_init(lin):
    nn.init.normal_(lin.weight, std=0.01)
    nn.init.zeros_(lin.bias)

def xavier_init(lin):
    nn.init.xavier_normal_(lin.weight)
    nn.init.zeros_(lin.bias)

def he_init(lin):
    nn.init.kaiming_normal_(lin.weight, nonlinearity="relu")
    nn.init.zeros_(lin.bias)

for name, fn in [("small", small_init), ("xavier", xavier_init), ("he", he_init)]:
    stds = track_activation_std(fn)
    print(name, "first", round(stds[0], 3), "last", round(stds[-1], 3))
